In [67]:
import pandas as pd
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import json
import csv
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .enableHiveSupport()
    .appName("cu2")
    .master("yarn")
    .config("spark.submit.deployMode", "client")
    .getOrCreate()
)
sql = spark.sql


In [68]:
df = pd.read_csv(
    "/data/scratch/ldedieu/CU2/encoder_baseline/MISTRAL/train_format2.csv",
    #quoting=csv.QUOTE_NONE,
    on_bad_lines='skip'
)

In [69]:
len(df)

10499

In [70]:
df = df.dropna()
len(df)

9356

In [71]:
df.columns

Index(['index', 'clinical_note', 'procedure_json', 'annot_diagnostics_v2'], dtype='str')

In [73]:
df = df[["clinical_note", "annot_diagnostics_v2"]]

In [74]:
import pandas as pd
import ast

def extract_all_diagnostics(cell_value):
    # Initialisation des résultats par défaut
    res = {'dp': None, 'dr': None, 'das': []}
    
    if pd.isna(cell_value):
        return pd.Series(res)
    
    if isinstance(cell_value, str):
        try:
            cell_value = ast.literal_eval(cell_value)
        except (ValueError, SyntaxError):
            return pd.Series(res)
            
    if isinstance(cell_value, dict):
        for key, details in cell_value.items():
            if isinstance(details, dict):
                pos = details.get('position')
                if pos == 'DP':
                    res['dp'] = key
                elif pos == 'DR':
                    res['dr'] = key
                elif pos == 'DAS':
                    res['das'].append(key)
                    
    return pd.Series(res)

# Application sur la colonne pour créer les 3 nouvelles colonnes en une seule passe
df[['dp', 'dr', 'das']] = df['annot_diagnostics_v2'].apply(extract_all_diagnostics)
df['das'] = df['das'].apply(lambda x: list(set(x)) if isinstance(x, list) else x)
df = df.drop("annot_diagnostics_v2", axis=1)

In [75]:
df

,clinical_note,dp,dr,das
0,Nouvel Hôpital Civil - Hôpitaux Universitaires...,E86,None,"[C445, I5002, I10, B9680, Z922, M316, Z803]"
1,Hôpital Nord - Assistance Publique – Hôpitaux ...,L030,None,"[Q796, Y98, B954, D66, C414, Z292]"
2,"M. Sasa Fauche, 60 ans, est admis en hospitali...",C61,None,"[I10, Z950, R5218, E220, C772, M8885, F102]"
3,Hôpital Saint-Louis - Assistance Publique Hôpi...,I352,None,"[I440, Z713, C787, D472, Z952, F608, M1099, C7..."
4,Service de Pédiatrie Générale – Hôpital Pelleg...,G478,None,[]
...,...,...,...,...
10494,SERVICE DE RADIOTHÉRAPIE\nHôpital Larrey - Cen...,C797,None,"[C61, R64, C786, I10, F1026, Z515, Z998, C795,..."
10495,"Madame Nafi Picot, 52 ans, est hospitalisée le...",C187,None,"[I258, K20, Z960, F500, N871, R572, Z991+, G31..."
10496,HÔPITAL DE LA CROIX-ROUSSE - HOSPICES CIVILS D...,T402,None,"[F2000, T455, E6605, F4322]"
10497,Hôpital Bichat-Claude Bernard - Assistance Pub...,S05,None,"[F20, V49]"


In [76]:
df['mdp'] = 'Z769'

mask = df['dr'].notna()

df.loc[mask, 'mdp'] = df.loc[mask, 'dp']
df.loc[mask, 'dp'] = df.loc[mask, 'dr']

df.drop(columns=['dr'], inplace=True)

In [77]:
df

,clinical_note,dp,das,mdp
0,Nouvel Hôpital Civil - Hôpitaux Universitaires...,E86,"[C445, I5002, I10, B9680, Z922, M316, Z803]",Z769
1,Hôpital Nord - Assistance Publique – Hôpitaux ...,L030,"[Q796, Y98, B954, D66, C414, Z292]",Z769
2,"M. Sasa Fauche, 60 ans, est admis en hospitali...",C61,"[I10, Z950, R5218, E220, C772, M8885, F102]",Z769
3,Hôpital Saint-Louis - Assistance Publique Hôpi...,I352,"[I440, Z713, C787, D472, Z952, F608, M1099, C7...",Z769
4,Service de Pédiatrie Générale – Hôpital Pelleg...,G478,[],Z769
...,...,...,...,...
10494,SERVICE DE RADIOTHÉRAPIE\nHôpital Larrey - Cen...,C797,"[C61, R64, C786, I10, F1026, Z515, Z998, C795,...",Z769
10495,"Madame Nafi Picot, 52 ans, est hospitalisée le...",C187,"[I258, K20, Z960, F500, N871, R572, Z991+, G31...",Z769
10496,HÔPITAL DE LA CROIX-ROUSSE - HOSPICES CIVILS D...,T402,"[F2000, T455, E6605, F4322]",Z769
10497,Hôpital Bichat-Claude Bernard - Assistance Pub...,S05,"[F20, V49]",Z769


In [78]:
df = df[df['dp'].apply(lambda x: isinstance(x, str))]
len(df)

9334

In [79]:
codes = pd.read_csv(
    "../data/Referentiel_CIM-10-20250108.csv", sep=";", header=1, dtype=str
)

codes = codes[["code", "libellé long", "code MCO/HAD"]].rename(
    columns={"libellé long": "definition", "code MCO/HAD": "mco_had"}
)
codes["code"] = codes["code"].astype(str).str.strip()
codes["mco_had"] = pd.to_numeric(codes["mco_had"], errors="coerce")

dp = codes[codes["mco_had"].isin([0])].drop("mco_had", axis=1)
das = codes[codes["mco_had"].isin([0,1,2])].drop("mco_had", axis=1)

dp=set(dp["code"].to_list())
das=set(das["code"].to_list())
print(len(dp))
print(len(das))

13358
40333


In [80]:
df = df[df['dp'].isin(dp)]
print(len(df))
df = df[df['mdp'].isin(dp)]
print(len(df))
df['das'] = df['das'].apply(
    lambda x: [code for code in x if code in das] if isinstance(x, list) else x
)
print(len(df))

8586
8586
8586


In [82]:
df_spark = spark.createDataFrame(df) \
    .withColumnRenamed("clinical_note", "note_text") 

In [83]:
df_spark.show(2,vertical=True)

-RECORD 0-------------------------
 note_text | Nouvel Hôpital Ci... 
 dp        | E86                  
 das       | [C445, I5002, I10... 
 mdp       | Z769                 
-RECORD 1-------------------------
 note_text | Hôpital Nord - As... 
 dp        | L030                 
 das       | [Q796, Y98, B954,... 
 mdp       | Z769                 
only showing top 2 rows



In [84]:
df = df_spark

In [85]:
df = df.filter(
    (F.col("dp").isNotNull()) & (F.col("dp") != "")
)

In [87]:
labels_dp = df.select("dp").distinct().rdd.flatMap(lambda x: x).collect()
labels_mdp = df.select("mdp").distinct().rdd.flatMap(lambda x: x).collect()
labels_das = df.select(F.explode("das")).distinct().rdd.flatMap(lambda x: x).collect()
print(len(labels_dp))
print(len(labels_mdp))
print(len(labels_das))

[Stage 13:===================================================>(1991 + 5) / 2000]

1925
1
4344


In [88]:
label_counts_dp = (
    df.groupBy("dp")
      .agg(F.count("*").alias("count"))
      .orderBy(F.desc("count"))
)

label_counts_mdp = (
    df.groupBy("mdp")
      .agg(F.count("*").alias("count"))
      .orderBy(F.desc("count"))
)

label_counts_das = (
    df.select(F.explode("das").alias("code_das"))
      .groupBy("code_das")
      .count()
      .orderBy(F.desc("count"))
)
print(label_counts_dp.count())
print(label_counts_mdp.count())
print(label_counts_das.count())
#label_counts_dp.show()

1925


1


[Stage 29:=================================================>  (1885 + 5) / 2000]

4344


In [89]:
label_counts_das.show(10)

[Stage 33:===============================================>    (1816 + 5) / 2000]

+--------+-----+
|code_das|count|
+--------+-----+
|     I10| 2949|
|    C787| 1280|
|    C795| 1271|
|    C780| 1137|
|    C786|  786|
|    C772|  608|
|    C771|  587|
|    Z515|  567|
|    C793|  563|
|    C770|  417|
+--------+-----+
only showing top 10 rows



In [91]:
from pyspark.sql import functions as F

def save_df_to_parquet(df, filename, num_partitions):
    df = df.withColumn("note_id", F.monotonically_increasing_id())

    df_to_save = df.select(
        "note_id",
        F.col("note_text"),
        F.col("dp"),
        F.col("das"),
        F.col("mdp"),
    )

    df_to_save.repartition(num_partitions).write.mode("overwrite").parquet(filename)


save_df_to_parquet(df, "CU2/encoder_baseline/MISTRAL/v1_das_mdp", num_partitions=9)


In [92]:
import subprocess
user="ldedieu"

subprocess.run(
    #["mkdir", "-p", f"/data/scratch/{user}/ia_codage/DP_DR/{strat}_{n_class_dp}class_v2"],
    ["mkdir", "-p", f"/data/scratch/{user}/CU2/encoder_baseline/MISTRAL/v1_das_mdp"],
    check=True
)

subprocess.run(
    [
        "hdfs", "dfs", "-copyToLocal", "-f",
        #f"ia_codage/dataset_800k/{strat}_{n_class_dp}class_v2/*",
        #f"/data/scratch/{user}/ia_codage/DP_DR/{strat}_{n_class_dp}class_v2/"
        f"CU2/encoder_baseline/MISTRAL/v1_das_mdp/*",
        f"/data/scratch/{user}/CU2/encoder_baseline/MISTRAL/v1_das_mdp/"
    ],
    check=True
)

CompletedProcess(args=['hdfs', 'dfs', '-copyToLocal', '-f', 'CU2/encoder_baseline/MISTRAL/v1_das_mdp/*', '/data/scratch/ldedieu/CU2/encoder_baseline/MISTRAL/v1_das_mdp/'], returncode=0)